# Movilidad - Barrio Obrero (Cali)

Mapa de la dimension **Movilidad** del ITT, formato estandar de Ciudad Paraiso:
puntos por categoria + mapa de calor (HeatMap), panel colapsable con serie
temporal 2023-2026 y el score OFICIAL de la dimension tomado del ITT maestro
(`itt_maestro_barrio_obrero.json`). Salida: `mapa_html/Barrio_obrero/Barrio_obrero_actual/mapa_movilidad_barrio_obrero.html`.

## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl','rasterio','pillow'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo')
else:
    print('Entorno local (uv).')


## 1. Setup, datos y helpers de mapa

In [ ]:
import os, json, math
import numpy as np, pandas as pd, geopandas as gpd
import folium
from folium.plugins import HeatMap
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_AMARILLO="#F0E442"
OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"; OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"; CRS_WEB=4326; LIMITE="#1B4F8A"
def _ruta(*cands):
    for p in cands:
        if os.path.isfile(p) or os.path.isdir(p): return os.path.abspath(p)
    return None
def _en(d,pref):
    if not d: return None
    for f in os.listdir(d):
        if f.startswith(pref) and f.endswith(".geojson"): return os.path.join(d,f)
    return None
DATADIR=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","barrio_obrero_completo"),
              os.path.join("data","Barrio_Obrero_geojson","barrio_obrero_completo"),
              os.path.join("/content","data","Barrio_Obrero_geojson","barrio_obrero_completo"))
POLY=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"),
           os.path.join("data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"),
           os.path.join("/content","data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"))
MAESTRO=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"),
              os.path.join("data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"),
              os.path.join("/content","data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"))
assert DATADIR and POLY and MAESTRO, "faltan datos de Barrio Obrero o el JSON maestro"
MASTER=json.load(open(MAESTRO,encoding="utf-8"))
POLIGONO_ZONA=gpd.read_file(POLY).to_crs(CRS_WEB)
_b=POLIGONO_ZONA.total_bounds; BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try: os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: pass
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero","Barrio_obrero_actual"))
os.makedirs(MAPAS_DIR,exist_ok=True)
def cargar_pts(pref, filtro_col=None, filtro_val=None):
    p=_en(DATADIR,pref)
    if p is None: return gpd.GeoDataFrame(geometry=[],crs=CRS_WEB)
    g=gpd.read_file(p).to_crs(CRS_WEB)
    g=g[g.geometry.notna() & g.geom_type.str.contains("Point")].copy()
    if filtro_col and filtro_col in g.columns:
        g=g[g[filtro_col].astype(str).str.strip().str.upper()==filtro_val].copy()
    if len(g):
        g["lat"]=g.geometry.y; g["lon"]=g.geometry.x
    else:
        g["lat"]=[]; g["lon"]=[]
    return g
def base_map(zoom=15):
    _c=POLIGONO_ZONA.geometry.union_all().centroid
    m=folium.Map(location=[_c.y,_c.x],zoom_start=zoom,tiles=None,control_scale=True,
                 zoom_snap=0.25,zoom_delta=0.1,max_zoom=22)
    m.get_root().width="100%"; m.get_root().height="100%"
    m.get_root().header.add_child(folium.Element(
        "<style>html,body{height:100%;margin:0;padding:0}"
        ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
        ".leaflet-top,.leaflet-bottom{z-index:1000}"
        "#dim-panel,#itt-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
        "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
        ".export-ctl{z-index:1002}</style>"))
    folium.TileLayer("https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",attr="Google",name="Google Streets",show=True,max_zoom=22,max_native_zoom=20).add_to(m)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",attr="Esri",name="Esri Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
    folium.TileLayer("https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",attr="Google",name="Google Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
    _zg=POLIGONO_ZONA.geometry.union_all(); _mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
    for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]):
        _ag.append([[x,y] for x,y in _gm.exterior.coords])
    folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
        name="Mascara (foco en la zona)",control=False,
        style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.18}).add_to(m)
    folium.GeoJson(POLIGONO_ZONA.to_json(),name="Limite Barrio Obrero",show=True,
        style_function=lambda f:{"color":LIMITE,"weight":2.5,"fill":False}).add_to(m)
    return m
def add_puntos(m, g, nombre, color, borde=None, show=False):
    fg=folium.FeatureGroup(name="Puntos: "+nombre+" ("+str(len(g))+")",show=show)
    bc=borde or color
    for _,r in g.iterrows():
        folium.CircleMarker([r["lat"],r["lon"]],radius=3,color=bc,weight=0.6,
            fill=True,fill_color=color,fill_opacity=0.75).add_to(fg)
    fg.add_to(m); return fg
def add_calor(m, g, nombre, grad, show=False):
    pts=g[["lat","lon"]].dropna().values.tolist() if len(g) else []
    fg=folium.FeatureGroup(name="Calor: "+nombre+" ("+str(len(pts))+")",show=show)
    if pts:
        HeatMap(pts,gradient=grad,radius=18,blur=14,min_opacity=0.35,name="heat").add_to(fg)
    fg.add_to(m); return fg
GRAD_AZUL={0:"#eff3ff",0.4:"#9ecae1",0.7:"#4292c6",1:"#08306b"}
GRAD_ROJO={0:"#fee5d9",0.4:"#fcae91",0.7:"#fb6a4a",1:"#67000d"}
GRAD_VERDE={0:"#edf8e9",0.4:"#a1d99b",0.7:"#41ab5d",1:"#00441b"}
GRAD_NARA={0:"#feedde",0.4:"#fdbe85",0.7:"#fd8d3c",1:"#a63603"}
GRAD_AMAR={0:"#ffffd4",0.4:"#fed98e",0.7:"#fe9929",1:"#993404"}
GRAD_MORA={0:"#f2f0f7",0.4:"#cbc9e2",0.7:"#9e9ac8",1:"#54278f"}
def _slider(m):
    mid=m.get_name()
    html=("<div id='heat-intensity-ctrl' style='position:fixed;bottom:24px;right:24px;z-index:1000;"
          "background:"+BG+";padding:8px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);"
          "font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:"+OKI_GRIS+";max-width:210px'>"
          "<div style='font-weight:700;margin-bottom:4px'>Intensidad del calor</div>"
          "<input id='heat-range' type='range' min='8' max='40' value='18' style='width:100%;cursor:pointer'>"
          "<div style='font-size:10px;color:#666;margin-top:2px'>Radio: <span id='heat-val'>18</span> px</div></div>"
          "<script>(function(){function ap(v){var mp=window['"+mid+"'];if(!mp||!mp.eachLayer)return;"
          "mp.eachLayer(function(l){if(l&&l.setOptions&&l._heat){try{l.setOptions({radius:+v,blur:Math.round(v*0.8)});l.redraw&&l.redraw();}catch(e){}}});}"
          "function init(){var r=document.getElementById('heat-range'),o=document.getElementById('heat-val');"
          "if(!r)return;r.addEventListener('input',function(){o.textContent=r.value;ap(r.value);});}"
          "if(document.readyState==='complete'){setTimeout(init,400);}else{window.addEventListener('load',function(){setTimeout(init,400);});}})();</script>")
    m.get_root().html.add_child(folium.Element(html))
print("Setup OK. Salida:",MAPAS_DIR)


## 2. Exportacion de lamina (deck)

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#dim-panel, #itt-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#dim-panel, #itt-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#dim-panel, #itt-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 3. Construccion del mapa

In [ ]:
m = base_map(zoom=15)
g_sin = cargar_pts("07_movilidad_siniestros")
add_calor(m, g_sin, "siniestros", GRAD_NARA, show=True)
add_puntos(m, g_sin, "siniestros", OKI_NARANJA, show=False)
_prp=_en(DATADIR,"11_red_peatonal_osm")
if _prp:
    gr=gpd.read_file(_prp).to_crs(CRS_WEB)
    folium.GeoJson(gr.to_json(),name="Red peatonal OSM (contexto)",show=False,
        style_function=lambda f:{"color":"#888","weight":1.2,"opacity":0.7}).add_to(m)
PANEL_TIT="Movilidad"; PANEL_DIMCOL="score_movilidad"
PANEL_RESUMEN='<div style="font-size:11px;margin:2px 0 4px">Siniestros: <b>%d</b> (en zona)</div>'%(len(g_sin))
PANEL_SERIE=[{"label":"Siniestralidad","col":"siniestralidad","hoja":"movilidad","color":OKI_NARANJA},
             {"label":"Lesionados","col":"lesionados","hoja":"movilidad","color":OKI_AZUL_C},
             {"label":"Mortales","col":"mortales","hoja":"movilidad","color":OKI_BERMELL}]
# Panel de dimension con serie 2023-2026 del maestro. Usa variables definidas
# por cada mapa: PANEL_TIT, PANEL_DIMCOL, PANEL_RESUMEN, PANEL_SERIE.
_res = MASTER["resumen"]
_anios = [str(r["año"]) for r in _res]
# Construir series de conteos reales desde las hojas del maestro.
_series_js = []
for s in PANEL_SERIE:
    hoja = MASTER[s["hoja"]]
    data = []
    for a in _anios:
        fila = [x for x in hoja if str(x["año"]) == a]
        v = fila[0].get(s["col"]) if fila else None
        data.append(v)
    _series_js.append({"label": s["label"], "data": data, "color": s["color"]})
_chart = json.dumps({"labels": _anios, "series": _series_js}, ensure_ascii=False)
# Filas: score oficial de la dimension por anio.
_filas = ""
for r in _res:
    _nt = " (parcial)" if "Parcial" in str(r["nivel"]) else ""
    _sc = r.get(PANEL_DIMCOL)
    _filas += ('<div style="display:flex;justify-content:space-between;font-size:11px;margin:1px 0">'
               '<span>%s%s</span><b>%s</b></div>' % (r["año"], _nt, ("%.1f" % _sc) if _sc is not None else "s/d"))
_panel = r"""
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="dim-panel" style="position:fixed;bottom:25px;left:20px;z-index:1001;max-width:400px;width:min(400px,44vw);max-height:84vh;overflow-y:auto;padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">__TITULO__ - Barrio Obrero</div>
    <button id="dim-toggle" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="dim-body">
    <div style="font-size:10px;color:#666;margin:4px 0 4px">Dimension del ITT oficial (CDO Alcaldia de Cali). Eventos reales en zona + score oficial por ano. 2026 parcial (no comparable).</div>
    __RESUMEN__
    <div style="margin:6px 0 2px;font-weight:700;font-size:11px">Score ITT de la dimension por ano:</div>
    __FILAS__
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">Serie de tiempo 2023-2026 / Ocultar</summary>
      <div style="position:relative;height:200px;width:100%;margin-top:6px"><canvas id="dim-chart"></canvas></div>
      <div style="font-size:9px;color:#666;margin-top:3px">Eje Y: Cantidad.</div>
    </details>
  </div>
</div>
<script>
(function(){
  var D=__CHART__;
  function dib(){
    var cv=document.getElementById("dim-chart"); if(!cv||typeof Chart==="undefined")return;
    new Chart(cv.getContext("2d"),{type:"line",
      data:{labels:D.labels,datasets:D.series.map(function(s){return {label:s.label,data:s.data,borderColor:s.color,backgroundColor:s.color,borderWidth:2,pointRadius:2,tension:0.25,fill:false};})},
      options:{responsive:true,maintainAspectRatio:false,interaction:{mode:"index",intersect:false},
        plugins:{legend:{labels:{boxWidth:10,font:{size:10}}}},
        scales:{x:{ticks:{font:{size:9}}},y:{beginAtZero:true,title:{display:true,text:"Cantidad",font:{size:9}},ticks:{font:{size:9}}}}}});
  }
  function initC(){var b=document.getElementById("dim-toggle"),bo=document.getElementById("dim-body");if(!b||!bo)return;b.addEventListener("click",function(){var o=(bo.style.display==="none");bo.style.display=o?"":"none";b.textContent=o?"_":"+";});}
  if(document.readyState==="complete"){setTimeout(dib,300);initC();}else{window.addEventListener("load",function(){setTimeout(dib,300);initC();});}
})();
</script>
"""
_panel = (_panel.replace("__GRIS__", OKI_GRIS).replace("__AZUL__", OKI_AZUL)
          .replace("__TITULO__", PANEL_TIT).replace("__RESUMEN__", PANEL_RESUMEN)
          .replace("__FILAS__", _filas).replace("__CHART__", _chart))
m.get_root().html.add_child(folium.Element(_panel))
print("Panel de dimension inyectado:", PANEL_TIT)

folium.LayerControl(collapsed=True,position="topright").add_to(m)
_slider(m)
print("Mapa movilidad OK")

## 4. Exportacion del HTML

In [ ]:
agregar_export(m, "Movilidad - Barrio Obrero", "mov_bo")
html=os.path.join(MAPAS_DIR,"mapa_movilidad_barrio_obrero.html")
m.save(html)
print("exportado:", html)
m